In [ ]:
!pip install librosa scikit-learn soundfile numpy

In [ ]:
import os
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score


In [ ]:
def extract_features(audio_path, sr=44100):

    signal, sr = librosa.load(audio_path, sr=sr)

    # =====================
    # MFCC (13)
    # =====================
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    # =====================
    # PLP
    # =====================

    # Espectrograma Mel
    mel_spec = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=26,
        power=2
    )

    # Compresión log
    log_mel = np.log(mel_spec + 1e-9)

    # DCT (similar a PLP)
    plp = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=13
    )

    plp_mean = np.mean(plp, axis=1)
    plp_std = np.std(plp, axis=1)

    # Concatenar
    features = np.concatenate([
        mfcc_mean, mfcc_std,
        plp_mean, plp_std
    ])

    return features


In [ ]:
emotion_map = {
    "alegria": "positivo",
    "neutro": "neutral",
    "tristeza": "negativo",
    "miedo": "negativo",
    "asco": "negativo",
    "sorpresa": "negativo",
    "ira": "negativo"
}

emotion2id = {
    "positivo": 0,
    "neutral": 1,
    "negativo": 2
}

def load_dataset(dataset_path):
    X, y = [], []

    print("Leyendo archivos desde:", dataset_path)

    for file in os.listdir(dataset_path):


        if not file.lower().endswith(".wav"):
            continue

        # Extraer emoción
        emotion_raw = os.path.splitext(file)[0].split("_")[1].lower()

        if emotion_raw not in emotion_map:
            print("Emoción no reconocida:", emotion_raw)
            continue

        label = emotion2id[emotion_map[emotion_raw]]

        filepath = os.path.join(dataset_path, file)

        features = extract_features(filepath)

        if features is None:
            print("Error extrayendo features:", file)
            continue

        X.append(features)
        y.append(label)

    print("Total muestras cargadas:", len(X))

    return np.array(X), np.array(y)

In [ ]:
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("===== MÉTRICAS GLOBALES =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))


Leyendo archivos desde: /content/dataset
Total muestras cargadas: 2001
===== MÉTRICAS GLOBALES =====
Accuracy : 0.5112
Precision: 0.5858
Recall   : 0.6970
F1-score : 0.5173
              precision    recall  f1-score   support

    positivo       0.26      0.88      0.41        51
     neutral       0.58      0.93      0.72        95
    negativo       0.91      0.28      0.43       255

    accuracy                           0.51       401
   macro avg       0.59      0.70      0.52       401
weighted avg       0.75      0.51      0.50       401



In [ ]:
#Carga del dataset
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

print("Muestras totales:", X.shape)

Leyendo archivos desde: /content/dataset
Total muestras cargadas: 2001
Muestras totales: (2001, 52)


In [ ]:
#Division de train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
#Modelo SVM
model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

SVC(C=10, class_weight='balanced')

In [ ]:
#Evaliación
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nReporte de clasificación:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))


Accuracy: 0.5112219451371571

Reporte de clasificación:

              precision    recall  f1-score   support

    positivo       0.26      0.88      0.41        51
     neutral       0.58      0.93      0.72        95
    negativo       0.91      0.28      0.43       255

    accuracy                           0.51       401
   macro avg       0.59      0.70      0.52       401
weighted avg       0.75      0.51      0.50       401

